# Day 08 · 하네스 — 다루기·확장·직접 만들기

**하네스 = 에이전트를 돌리는 런타임.** Claude Code를 해부하고(권한·컨텍스트·확장점), 스킬·플러그인·훅·MCP로 확장한 뒤, 마지막에 **미니 하네스**를 직접 조립한다.

- **Lab 1~4는 로컬 Claude Code** 환경에서 한다 (설치·VS Code 확장/데스크톱). 이 노트북의 코드 셀은 템플릿 파일을 `day08_artifacts/`에 만들어 눈으로 확인하는 용도.
- **Lab 5(미니 하네스)는 NVIDIA Qwen**으로 실제 동작한다.

## Lab 1 · 하네스 구조 투어 (로컬 Claude Code)

Claude Code를 열고 아래를 실제로 확인한다:

- **권한 모드** — 도구 실행을 자동/확인 중 무엇으로 할지
- **컨텍스트·컴팩션** — 길어지면 어떻게 요약되는지 (`/compact`)
- **서브에이전트** — 작업을 별도 에이전트에 위임
- **MCP** — `claude mcp list` 로 연결된 서버 확인
- **슬래시 커맨드** — `/help` 로 명령 목록
- **훅** — `.claude/settings.json` 의 hooks

> 7강에서 "런타임이 대신 해준 것"의 실체를 눈으로 확인하는 시간.

## Lab 2 · 스킬 만들기

스킬 = **필요할 때만 불러오는 노하우**. `description`이 트리거다 — 언제 쓸지 하네스가 그걸 보고 판단한다. 아래 셀은 예제 `SKILL.md`를 만든다.

In [ ]:
# Lab 2 — 스킬 하나를 SKILL.md로 작성한다 (여기선 day08_artifacts/ 에 만들어 눈으로 확인).
import os
os.makedirs("day08_artifacts/skills/diff-summary", exist_ok=True)
SKILL = '''---
description: 커밋 전 변경사항을 요약하고 위험요소를 짚는다. diff를 검토하거나 커밋 메시지를 쓸 때 사용.
allowed-tools: Bash(git *)
---

## 현재 변경
!`git diff HEAD`

## 지시
위 변경을 2~3줄로 요약하고, 위험요소(하드코딩 값·에러 처리 누락·테스트 미갱신)를 나열하라.
'''
open("day08_artifacts/skills/diff-summary/SKILL.md", "w", encoding="utf-8").write(SKILL)
print("작성: day08_artifacts/skills/diff-summary/SKILL.md")
print("→ 실제 사용: 이 폴더를  ~/.claude/skills/  또는  프로젝트 .claude/skills/  로 옮기면")
print("   description을 보고 Claude Code가 필요할 때 스스로 이 스킬을 불러온다.")

## Lab 3 · 플러그인으로 묶기

플러그인 = 스킬·커맨드·훅을 **한 패키지로** 묶어 배포·공유. 매니페스트는 `.claude-plugin/plugin.json`(필수).

In [ ]:
# Lab 3 — 스킬을 '플러그인'으로 묶는다. 매니페스트는 .claude-plugin/plugin.json (필수).
import os, json, shutil
base = "day08_artifacts/my-plugin"
os.makedirs(base + "/.claude-plugin", exist_ok=True)
os.makedirs(base + "/skills/diff-summary", exist_ok=True)
manifest = {"name": "my-plugin", "description": "강의용 예제 플러그인",
            "version": "1.0.0", "author": {"name": "student"}}
open(base + "/.claude-plugin/plugin.json", "w", encoding="utf-8").write(json.dumps(manifest, ensure_ascii=False, indent=2))
src = "day08_artifacts/skills/diff-summary/SKILL.md"          # Lab 2 스킬을 플러그인 안으로
if os.path.exists(src):
    shutil.copy(src, base + "/skills/diff-summary/SKILL.md")
print("플러그인 생성:")
for root, _, files in os.walk(base):
    for f in files:
        print("  ", os.path.join(root, f))
print("→ 로컬 테스트:  claude --plugin-dir", base)

## Lab 4 · 훅 + MCP 연결

**훅**은 도구 실행 전/후에 개입해 정책을 강제한다. 아래 셀은 Bash 실행 전 `rm -rf`를 막는 PreToolUse 훅을 만든다.

In [ ]:
# Lab 4 — PreToolUse 훅: Bash 실행 '전에' 스크립트를 통과시킨다 (정책 강제).
import os, json
os.makedirs("day08_artifacts/.claude/hooks", exist_ok=True)
guard = '''#!/bin/bash
INPUT=$(cat)                                   # 훅은 stdin으로 이벤트 JSON을 받는다
CMD=$(echo "$INPUT" | jq -r '.tool_input.command')
if echo "$CMD" | grep -q "rm -rf"; then
  echo "차단: rm -rf 는 허용되지 않습니다" >&2   # 이유는 stderr로
  exit 2                                        # exit 2 = 도구 실행 차단
fi
exit 0                                          # exit 0 = 통과
'''
open("day08_artifacts/.claude/hooks/guard.sh", "w", encoding="utf-8").write(guard)
settings = {"hooks": {"PreToolUse": [
    {"matcher": "Bash", "hooks": [
        {"type": "command", "command": "\"$CLAUDE_PROJECT_DIR\"/.claude/hooks/guard.sh"}]}]}}
open("day08_artifacts/.claude/settings.json", "w", encoding="utf-8").write(json.dumps(settings, ensure_ascii=False, indent=2))
print("작성: day08_artifacts/.claude/settings.json  +  hooks/guard.sh")
print("→ 실제 사용: 이 .claude/ 를 프로젝트 루트에 두면, Bash 실행 전 guard.sh 를 통과해야 한다.")

**MCP 연결** — 5강 예제 서버를 Claude Code에 등록한다 (터미널):

```bash
claude mcp add --transport stdio helper -- python day05/deploy/server.py
claude mcp list        # 등록 확인
```

프로젝트 공유용은 `.mcp.json`에 적는다:

```json
{ "mcpServers": { "helper": { "type": "stdio", "command": "python", "args": ["day05/deploy/server.py"] } } }
```

> 스킬 = 노하우, MCP = 도구 — 역할이 다르다.

## Lab 5 · 미니 하네스 직접

하네스의 네 조각을 직접 조립한다: **도구 레지스트리 + 에이전트 루프(7강) + 컨텍스트 트리밍 + 확장점(스킬)**. (NVIDIA Qwen 필요)

In [ ]:
%pip install -q openai

In [ ]:
# Lab 5(미니 하네스)에서 LLM으로 Qwen을 부른다. 토큰은 입력창/환경변수 (하드코딩 금지).
import os, getpass, json
from openai import OpenAI

LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in llm.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank", "thinking"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("모델:", LLM_MODEL)

In [ ]:
# Lab 5 — 미니 하네스 = 도구 레지스트리 + 에이전트 루프 + 컨텍스트 트리밍 + 확장점(스킬)
class MiniHarness:
    """'미니 Claude Code' — 하네스의 네 조각을 직접 조립한다."""
    def __init__(self, system, keep=10):
        self.system = system
        self.registry = {}      # 이름 → 함수
        self.specs = []         # OpenAI 도구 스펙
        self.skills = {}        # 확장점: 트리거 단어 → 노하우
        self.keep = keep        # 컨텍스트로 남길 최근 메시지 수

    def tool(self, description, params):                 # 도구 등록 데코레이터
        def deco(fn):
            self.registry[fn.__name__] = fn
            self.specs.append({"type": "function", "function": {
                "name": fn.__name__, "description": description,
                "parameters": {"type": "object",
                               "properties": {k: {"type": v} for k, v in params.items()},
                               "required": list(params)}}})
            return fn
        return deco

    def skill(self, trigger, text):                      # 확장점: 트리거 단어가 나오면 노하우 주입
        self.skills[trigger] = text

    def _system(self, q):
        extra = [t for k, t in self.skills.items() if k in q]
        return self.system + ("\n\n[스킬]\n" + "\n".join(extra) if extra else "")

    def _trim(self, messages):                           # 컨텍스트 트리밍: system + 최근 keep개
        if len(messages) <= self.keep + 1:
            return messages
        return messages[:1] + messages[-self.keep:]

    def run(self, question, max_steps=6):
        messages = [{"role": "system", "content": self._system(question)},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=self._trim(messages), tools=self.specs,
                temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  [도구] {tc.function.name}({args})")
                try:
                    out = self.registry[tc.function.name](**args)
                except Exception as e:
                    out = f"오류: {e}"
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(out, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)"

In [ ]:
# 미니 하네스에 도구 2개를 등록하고, 스킬(확장점)을 하나 붙여 돌려본다
harness = MiniHarness("너는 도구를 쓰는 조수다. 한국어로 간결히 답하라.")

@harness.tool(description="두 정수를 더한다", params={"a": "integer", "b": "integer"})
def add(a, b):
    return a + b

@harness.tool(description="도시의 (예시) 날씨를 반환한다", params={"city": "string"})
def weather(city):
    return {"city": city, "temp": 21, "sky": "맑음"}

harness.skill("날씨", "날씨를 물으면 weather 도구를 쓰고 섭씨로 답한다.")   # 확장점(스킬) 주입

print("A:", harness.run("서울 날씨 알려주고, 3 더하기 4도 계산해줘"))

## Lab · (Claude Code) 하네스 프레임워크 체험

스킬·플러그인의 실제 사례 — 유명 프레임워크를 설치해 기본 Claude Code와 비교한다. **로컬 Claude Code 터미널**에서 진행(노트북 커널 아님).

```bash
# Superpowers — TDD·서브에이전트 개발 methodology (가장 인기 플러그인)
/plugin marketplace add obra/superpowers
/plugin install superpowers
```

**해볼 것**
- 같은 작업을 기본 Claude Code vs 프레임워크로 시켜보고 흐름(서브에이전트·리뷰·역할) 차이를 관찰
- 토큰 절감 스킬 `ponytail`(YAGNI·꼭 필요한 코드만)·`caveman`(출력 압축)을 붙여 실제 절감을 확인

> ⚠️ caveman은 광고(65퍼센트)와 실측(~8.5퍼센트, 출력 토큰만)이 다르다 — **직접 재보는 것**이 실습의 핵심.

## 산출물 체크리스트

- [ ] Claude Code의 권한·컨텍스트·MCP·훅을 눈으로 확인했다
- [ ] 내 스킬(`SKILL.md`)을 만들었다 — description이 트리거
- [ ] 스킬을 플러그인(`.claude-plugin/plugin.json`)으로 묶었다
- [ ] PreToolUse 훅으로 위험 명령을 막았다 + 예제 MCP를 연결했다
- [ ] 미니 하네스를 조립해 돌렸다 (레지스트리·루프·트리밍·스킬)